In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("comments") \
    .config("spark.executor.instances", "6") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "120") \
    .getOrCreate()

print(spark.version)


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
7,application_1740181201664_0009,pyspark,idle,Link,Link,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

3.0.1-amzn-0

In [2]:
import json
# import pandas as pd
from pyspark.sql.functions import col, udf, to_timestamp, desc, year, count, round, monotonically_increasing_id, concat, lit
from pyspark.sql.functions import from_unixtime, from_json, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
import pyspark.sql.functions as F

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
posts = spark.read.parquet("s3://1313131buckey/politics_submissions.parquet")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
posts.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- title: string (nullable = true)
 |-- author: string (nullable = true)
 |-- id: string (nullable = true)
 |-- link_flair_text: string (nullable = true)
 |-- url: string (nullable = true)
 |-- time: string (nullable = true)

In [5]:
parent_id = posts.select(F.concat(F.lit("t3_"), posts["id"]).alias("parent_id"))
parent_id.show(10)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+
|parent_id|
+---------+
| t3_2gw58|
| t3_2hz89|
| t3_2i1ga|
| t3_2i46c|
| t3_2jgo8|
| t3_2jv8e|
| t3_2n1pf|
| t3_2non6|
| t3_2ovm9|
| t3_2qect|
+---------+
only showing top 10 rows

In [6]:
z = 's3://1313131buckey/politics_comments.zst'

data = spark.read.text(z)
data.first()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Row(value='{"body":"DAMN, she\'s a fox! Also I love his retort to the \\"association with Alex Jones\\" question.","controversiality":0,"stickied":false,"link_id":"t3_2cnnc","subreddit_id":"t5_2cneq","subreddit":"politics","created_utc":1186380318,"author_flair_css_class":null,"score":6,"ups":6,"author_flair_text":null,"author":"lojomofo","id":"c2cnq9","edited":false,"parent_id":"t3_2cnnc","gilded":0,"retrieved_on":1473762425,"distinguished":null}')

In [8]:
# schema for data
json_schema = StructType([
    StructField("body", StringType(), True),
    StructField("author", StringType(), True),
    StructField("created_utc", StringType(), True),
    StructField("score", IntegerType(), True),
    StructField("id", StringType(), True),
    StructField("parent_id", StringType(), True),
    StructField("link_id", StringType(), True),
    StructField("subreddit", StringType(), True),
    StructField("author_flair_text", StringType(), True)
])

json_comments = data.select(from_json(col("value"), json_schema).alias("mapped_comments"))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
filtered_comments = json_comments.join(
    parent_id,
    json_comments.mapped_comments.link_id == parent_id.parent_id,
    "inner"
).select("mapped_comments.*")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
filtered_comments = filtered_comments.withColumn(
    "time",
    from_unixtime(col("created_utc").cast("double").cast("bigint"))
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
filtered_comments.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------------+-----------+-----+------+---------+--------+---------+-----------------+-------------------+
|                body|        author|created_utc|score|    id|parent_id| link_id|subreddit|author_flair_text|               time|
+--------------------+--------------+-----------+-----+------+---------+--------+---------+-----------------+-------------------+
|Yes its true, ron...|     remembery| 1187624037|   -7|c2gw9z| t3_2gw58|t3_2gw58| politics|             null|2007-08-20 15:33:57|
|News flash, two o...|captainhaddock| 1187625718|    5|c2gwku| t3_2gw58|t3_2gw58| politics|             null|2007-08-20 16:01:58|
|I don't understan...|         jk3us| 1187625741|    2|c2gwkx| t3_2gw58|t3_2gw58| politics|             null|2007-08-20 16:02:21|
|Paul says all act...|      FrancisC| 1187648889|    3|c2h1pb| t3_2gw58|t3_2gw58| politics|             null|2007-08-20 22:28:09|
|           [deleted]|     [deleted]| 1187664634|    0|c2h3yl| t3_2gw58|t3_2gw58| politics

In [12]:
filtered_comments.filter(col("time").isNull()).show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+------+-----------+-----+---+---------+-------+---------+-----------------+----+
|body|author|created_utc|score| id|parent_id|link_id|subreddit|author_flair_text|time|
+----+------+-----------+-----+---+---------+-------+---------+-----------------+----+
+----+------+-----------+-----+---+---------+-------+---------+-----------------+----+

In [13]:
filtered_comments = filtered_comments.repartition(20) 
print(filtered_comments.rdd.getNumPartitions())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

1

In [14]:
filtered_comments.write.parquet("s3://1313131buckey/politics_commentsB.parquet")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
filtered_comments.printSchema()
print('Total Rows: %d' % filtered_comments.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…